# PA2 통합 제출 노트북

Level 1~5를 순서대로 실행합니다. Kaggle 제출 단계는 별도 공지에 따라 제외했습니다.


In [ ]:
import os
import sys

# 제출 기본값: 모든 모델을 처음부터 학습. 기존 checkpoint 평가만 할 때 False.
RUN_ALL_TRAINING = True
ENABLE_WANDB = False
os.environ["AUE8088_RUN_TRAINING"] = "1" if RUN_ALL_TRAINING else "0"
os.environ["AUE8088_USE_WANDB"] = "1" if ENABLE_WANDB else "0"

repo_url = "https://github.com/min0712-cdl/HYU-AUE8088-PA2.git"
repo_name = "HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone {repo_url}
else:
    !git -C /content/{repo_name} pull
%cd /content/{repo_name}

from google.colab import drive
drive.mount("/content/drive")
ARTIFACT_ROOT = os.environ.get("AUE8088_ARTIFACT_ROOT", "/content/drive/MyDrive/AUE8088_PA2")
CHECKPOINT_DIR = os.path.join(ARTIFACT_ROOT, "checkpoints")
OUTPUT_DIR = os.path.join(ARTIFACT_ROOT, "outputs")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

%load_ext autoreload
%autoreload 2
!pip install -q -r requirements.txt


# Level 1 — Classic CNNs (VGG-16, ResNet-18 / 50)

**목표**: VGG 와 ResNet 을 직접 구현하여 Set A 위에서 Multi-task 로 학습하고, **Skip Connection** 이 깊은 네트워크의 수렴에 미치는 영향을 분석합니다.

**금지 사항**: `torchvision.models.*`, `timm`. 단, 위 라이브러리의 코드를 *참고하여 직접 타이핑* 하는 것은 허용됩니다.

본 노트북의 산출물:
1. 학습된 체크포인트 (`checkpoints/level1_*.pth`).
2. VGG-16, ResNet-18, ResNet-50 의 `Avg-Macro-F1` 및 속성별 Macro-F1 표.
3. VGG (skip 없음) vs ResNet (skip 있음) 의 학습 손실 곡선 비교.

In [ ]:
import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, evaluation_report, print_evaluation_report
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.resnet import resnet18, resnet50
from src.models.vgg import VGG16

SEED = 42
set_seed(SEED, deterministic=True)   # 재현성을 위한 시드 고정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

### Weights & Biases 설정

학습 곡선과 필수 평가 메트릭을 로깅하려면 `USE_WANDB = True`로 바꾸세요. 기본값은 외부 평가에서 로그인이 중단되지 않도록 `False`입니다.

최초 1회만:
```python
USE_WANDB = True  # 다음 설정 셀에서만 로그인 실행
```

In [ ]:
USE_WANDB = os.environ.get("AUE8088_USE_WANDB", "0") == "1"
if USE_WANDB:
    import wandb
    wandb.login()

In [ ]:
WANDB_PROJECT = "aue8088-pa2" if USE_WANDB else None
WANDB_TAGS    = ["level1"]
RUN_TRAINING = os.environ.get("AUE8088_RUN_TRAINING", "1") == "1"  # False이면 checkpoint 평가만 함

In [ ]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# Set A 의 train/val 로더 구성
train_ds = BDDAttrDataset(DATA_ROOT, split="train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, split="val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# 속성별 클래스 분포 출력 — 불균형 정도를 직접 확인할 것
for a in ATTRIBUTES:
    print(f"{a:10s} 클래스별 샘플 수 (train) = {train_ds.class_counts(a).tolist()}")

In [ ]:
from torch import nn

def train_one(model_fn, name, epochs=20, loss_weights=None):
    """단일 모델을 학습하고 체크포인트와 wandb 산출물을 저장한다."""
    set_seed(SEED, deterministic=True)
    model = model_fn().to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"level1_{name}.pth")
    cfg = TrainConfig(
        epochs=epochs, checkpoint_path=checkpoint_path,
        checkpoint_metadata={"backbone": name},
    )
    if loss_weights is not None:
        cfg.loss_weights = loss_weights

    # wandb 로거 — WANDB_PROJECT 가 None 이거나 wandb 미설치 시 자동 no-op
    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level1-{name}",
        config={
            "backbone": name, "epochs": epochs, "batch": BATCH,
            "lr": 3e-4, "weight_decay": 5e-4, "seed": SEED,
            "loss_weights": cfg.loss_weights,
        },
        tags=WANDB_TAGS + [name],
    )

    trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger)
    history = trainer.fit(train_loader, val_loader)

    # trainer.fit은 정해진 epoch를 모두 수행한 뒤 validation 최고 가중치를 복원한다.
    val_pred, val_probs, val_tgt, _ = collect_predictions(model, val_loader, device)
    report = evaluation_report(val_pred, val_probs, val_tgt)
    print_evaluation_report(report, title=f"Level 1 - {name}")
    logger.log_evaluation("final", report)
    logger.finish()

    torch.save({
        "state_dict": model.state_dict(), "history": history,
        "best_val_avg_mf1": history["best_val_avg_mf1"],
        "best_epoch": history["best_epoch"], "backbone": name,
    },
               checkpoint_path)
    model = model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return model, history

def load_one(model_fn, name):
    path = os.path.join(CHECKPOINT_DIR, f"level1_{name}.pth")
    payload = torch.load(path, map_location="cpu")
    model = model_fn().to(device)
    model.load_state_dict(payload["state_dict"])
    val_pred, val_probs, val_tgt, _ = collect_predictions(model, val_loader, device)
    report = evaluation_report(val_pred, val_probs, val_tgt)
    print_evaluation_report(report, title=f"Level 1 - {name} (checkpoint)")
    model = model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return model, payload.get("history", {})

# TODO: 각 모델을 학습하고 최종 Avg-Macro-F1 및 속성별 MF1 을 기록하세요.
if RUN_TRAINING:
    vgg_model, vgg_hist = train_one(VGG16,    "vgg16",    epochs=30)
    r18_model, r18_hist = train_one(resnet18, "resnet18", epochs=30)
    r50_model, r50_hist = train_one(resnet50, "resnet50", epochs=30)
    r18_w211_model, r18_w211_hist = train_one(
        resnet18, "resnet18-w211", epochs=30,
        loss_weights={"weather": 2.0, "scene": 1.0, "timeofday": 1.0},
    )
else:
    vgg_model, vgg_hist = load_one(VGG16, "vgg16")
    r18_model, r18_hist = load_one(resnet18, "resnet18")
    r50_model, r50_hist = load_one(resnet50, "resnet50")
    r18_w211_model, r18_w211_hist = load_one(resnet18, "resnet18-w211")

## 분석 (리포트 필수 포함 항목)

1. **수렴 비교**: VGG-16 과 ResNet-18 의 학습 손실 곡선을 함께 그리세요. Skip connection 이 없는 깊은 네트워크가 정체되는 현상이 관찰되는지, 그 원인은 무엇인지 서술하세요.
2. **속성별 MF1 표**: 각 백본에 대해 Weather / Scene / Time of Day 의 Macro-F1 을 표로 정리하세요. 어느 속성이 가장 어려운지, 깊이 변화에 따라 그 양상이 어떻게 바뀌는지 분석하세요.
3. **Loss 가중치 민감도**: ResNet-18 에 대해 최소 두 가지 비자명한 가중치 설정 (예: `(1, 1, 1)` vs `(2, 1, 1)`) 으로 재학습하고 Avg-MF1 변화를 보고하세요.

wandb 를 활성화한 경우 같은 프로젝트의 Run 들을 비교하면 학습 곡선·속성별 MF1·confusion matrix 가 한눈에 정리됩니다.

In [ ]:
# 다음 Level을 위해 CPU/GPU 메모리를 정리합니다.
import gc
for _name in ["vgg_model","r18_model","r50_model","r18_w211_model","train_loader","val_loader","train_ds","val_ds"]:
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# Level 2 — Vision Transformers (ViT-S/16, 선택적으로 Swin-Tiny)

**목표**: ViT (와 선택적으로 Swin-T) 를 직접 구현하고 Multi-task 로 연결하여 Level 1 의 CNN 들과 비교합니다.

**Pretrained 가중치**: ImageNet `.pth` 파일을 본인이 구현한 모델의 `state_dict` 에 로드하는 것은 허용됩니다. 출처를 명시하세요. **`timm` / `torchvision.models` import 는 금지** 입니다.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, evaluation_report, print_evaluation_report
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.vit import vit_small_patch16_224
# from src.models.swin import SwinTiny  # 선택 사항

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
USE_WANDB = os.environ.get("AUE8088_USE_WANDB", "0") == "1"
if USE_WANDB:
    import wandb
    wandb.login()

WANDB_PROJECT = "aue8088-pa2" if USE_WANDB else None
WANDB_TAGS    = ["level2"]
RUN_TRAINING = os.environ.get("AUE8088_RUN_TRAINING", "1") == "1"  # False이면 checkpoint 평가만 함

In [ ]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# 선택 사항: 본인 ViT 구현체에 ImageNet pretrained 가중치를 로드하는 절차
#
# 진행 방식:
#   1) 공개된 ViT-S/16 체크포인트 (.pth) 를 다운로드.
#   2) 모델 인스턴스 생성:  model = vit_small_patch16_224()
#   3) 키 매핑 후 로드:
#        pre = torch.load('vit_s16.pth')
#        missing, unexpected = model.load_state_dict(remap(pre), strict=False)
#        # Multi-task head 는 task 종속이므로 random init 유지.
#
# 사용한 체크포인트 출처와 매칭된 키 개수를 리포트에 기재하세요.
DEIT_SMALL_URL = (
    "https://dl.fbaipublicfiles.com/deit/"
    "deit_small_patch16_224-cd65a155.pth"
)

def build_vit(use_pretrained):
    model = vit_small_patch16_224()
    info = {"source": None, "matched_tensors": 0, "matched_parameters": 0}

    if not use_pretrained:
        return model.to(device), info

    checkpoint = torch.hub.load_state_dict_from_url(
        DEIT_SMALL_URL, map_location="cpu", check_hash=True
    )
    pretrained = checkpoint["model"]
    target = model.state_dict()
    remapped = {}

    for key, value in pretrained.items():
        key = key.replace(".mlp.fc1.", ".mlp.0.")
        key = key.replace(".mlp.fc2.", ".mlp.3.")
        if key.startswith("head."):
            continue
        if key in target and target[key].shape == value.shape:
            remapped[key] = value

    missing, unexpected = model.load_state_dict(remapped, strict=False)
    non_head_missing = [key for key in missing if not key.startswith("head.")]
    if non_head_missing or unexpected:
        raise RuntimeError(
            f"Pretrained backbone mismatch: missing={non_head_missing}, "
            f"unexpected={unexpected}"
        )

    info = {
        "source": DEIT_SMALL_URL,
        "matched_tensors": len(remapped),
        "matched_parameters": sum(value.numel() for value in remapped.values()),
    }
    print(f"Matched pretrained tensors: {info['matched_tensors']}")
    print(f"Matched pretrained parameters: {info['matched_parameters']:,}")
    print(f"Random-init task head keys: {missing}")
    return model.to(device), info

In [ ]:
epochs = 30

def train_vit(use_pretrained):
    set_seed(SEED, deterministic=True)
    g.manual_seed(SEED)
    model, pretrain_info = build_vit(use_pretrained)
    optim = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    variant = "pretrained" if use_pretrained else "scratch"
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"level2_vit_s16_{variant}.pth")

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level2-vit_s16-{variant}",
        config={
            "backbone": "vit_s16", "pretrained": use_pretrained,
            "pretrained_source": pretrain_info["source"],
            "matched_tensors": pretrain_info["matched_tensors"],
            "matched_parameters": pretrain_info["matched_parameters"],
            "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
        },
        tags=WANDB_TAGS + ["vit_s16", variant],
    )
    trainer = MultiTaskTrainer(
        model, optim, sched, losses, device,
        TrainConfig(
            epochs=epochs, checkpoint_path=checkpoint_path,
            checkpoint_metadata={"pretrained": use_pretrained, "pretrained_info": pretrain_info},
        ), logger=logger
    )

    # TODO: trainer.fit(train_loader, val_loader)
    history = trainer.fit(train_loader, val_loader)

    # trainer.fit은 정해진 epoch를 모두 수행한 뒤 validation 최고 가중치를 복원한다.
    val_pred, val_probs, val_tgt, _ = collect_predictions(model, val_loader, device)
    report = evaluation_report(val_pred, val_probs, val_tgt)
    print_evaluation_report(report, title=f"Level 2 - ViT-S {variant}")
    logger.log_evaluation("final", report)
    logger.finish()
    torch.save(
        {"state_dict": model.state_dict(), "history": history,
         "best_val_avg_mf1": history["best_val_avg_mf1"],
         "best_epoch": history["best_epoch"],
         "pretrained": use_pretrained, "pretrained_info": pretrain_info},
        checkpoint_path,
    )
    model = model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return model, history

def load_vit(variant):
    path = os.path.join(CHECKPOINT_DIR, f"level2_vit_s16_{variant}.pth")
    payload = torch.load(path, map_location="cpu")
    state_dict = payload["state_dict"] if "state_dict" in payload else payload
    history = payload.get("history", {}) if "state_dict" in payload else {}
    model = vit_small_patch16_224().to(device)
    model.load_state_dict(state_dict)
    val_pred, val_probs, val_tgt, _ = collect_predictions(model, val_loader, device)
    report = evaluation_report(val_pred, val_probs, val_tgt)
    print_evaluation_report(report, title=f"Level 2 - ViT-S {variant} (checkpoint)")
    return model, history

if RUN_TRAINING:
    scratch_model, scratch_history = train_vit(use_pretrained=False)
    pretrained_model, pretrained_history = train_vit(use_pretrained=True)
else:
    scratch_model, scratch_history = load_vit("scratch")
    pretrained_model, pretrained_history = load_vit("pretrained")

## 분석 (리포트 필수 포함 항목)

1. **CNN vs Transformer**: 동일 epoch 예산 하에서 ResNet-50 (Level 1) 과 ViT-S (Level 2) 의 Avg-MF1 을 비교하세요.
2. **Pretrained vs Scratch**: 약 5천 장 규모의 소규모 데이터셋에서 ImageNet 초기화가 실제로 얼마나 도움이 되는지 정량적으로 보고하세요.
3. **속성별 거동**: ViT 가 ResNet 대비 Weather 와 Time of Day 사이의 오류 분포를 다르게 가져가는지, 그 원인을 가설로 제시하세요.

In [ ]:
# 다음 Level을 위해 CPU/GPU 메모리를 정리합니다.
import gc
for _name in ["scratch_model","pretrained_model","train_loader","val_loader","train_ds","val_ds"]:
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# Level 3 — 불균형 대응 및 고급 Augmentation

**목표**: 다수 클래스의 정확도를 크게 희생하지 않으면서, 소수 클래스 (foggy / snowy / dawn-dusk) 의 성능을 끌어올립니다.

다음 축에서 **최소 2가지 이상** 의 기법을 적용하세요.
- Loss-level: Weighted CE, Focal Loss, LDAM, Class-Balanced Loss
- Sampling-level: class-balanced sampler
- Augmentation-level: RandAugment, Mixup, CutMix

Level 1 / 2 에서 가장 좋았던 백본을 base 로 사용하세요. wandb 를 사용하면 여러 기법의 비교 Run 을 같은 프로젝트에 모아 볼 수 있어 편리합니다.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, evaluation_report, print_evaluation_report
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.datasets.samplers import class_balanced_sampler
from src.losses.imbalanced import FocalLoss, ClassBalancedLoss, LDAMLoss, weighted_cross_entropy
from src.augment.mix import mixup_data, cutmix_data, mixed_loss
from src.models.vit import vit_small_patch16_224

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
USE_WANDB = os.environ.get("AUE8088_USE_WANDB", "0") == "1"
if USE_WANDB:
    import wandb
    wandb.login()

WANDB_PROJECT = "aue8088-pa2" if USE_WANDB else None
WANDB_TAGS    = ["level3"]
RUN_TRAINING = os.environ.get("AUE8088_RUN_TRAINING", "1") == "1"  # False이면 checkpoint 평가만 함

In [ ]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
DEIT_SMALL_URL = (
    "https://dl.fbaipublicfiles.com/deit/"
    "deit_small_patch16_224-cd65a155.pth"
)

def build_pretrained_vit():
    model = vit_small_patch16_224()
    checkpoint = torch.hub.load_state_dict_from_url(
        DEIT_SMALL_URL, map_location="cpu", check_hash=True
    )
    pretrained = checkpoint["model"]
    target = model.state_dict()
    remapped = {}

    for key, value in pretrained.items():
        key = key.replace(".mlp.fc1.", ".mlp.0.")
        key = key.replace(".mlp.fc2.", ".mlp.3.")
        if key.startswith("head."):
            continue
        if key in target and target[key].shape == value.shape:
            remapped[key] = value

    missing, unexpected = model.load_state_dict(remapped, strict=False)
    non_head_missing = [key for key in missing if not key.startswith("head.")]
    if non_head_missing or unexpected:
        raise RuntimeError(
            f"Pretrained backbone mismatch: missing={non_head_missing}, "
            f"unexpected={unexpected}"
        )

    print(f"Matched pretrained tensors: {len(remapped)}")
    return model.to(device), len(remapped)

EPOCHS = 25
LR = 5e-4
WEIGHT_DECAY = 5e-2

EXPERIMENTS = [
    {"name": "baseline", "use_sampler": False, "use_mix": False},
    {"name": "sampler-only", "use_sampler": True, "use_mix": False},
    {"name": "mixup-cutmix-only", "use_sampler": False, "use_mix": True},
    {"name": "sampler+mixup-cutmix", "use_sampler": True, "use_mix": True},
]

In [ ]:
# 옵션 C — 학습 루프에 Mixup/CutMix 를 통합하여 적용
# (깨끗한 실험을 위해서는 _train_one_epoch 를 서브클래싱하는 것이 좋으나,
#  아래는 augmented step 의 핵심만 인라인으로 보인 것입니다.)

from tqdm import tqdm

def step_with_mix(model, loss_fns, images, targets):
    """50% 확률로 Mixup, 나머지 50% 확률로 CutMix 적용."""
    if torch.rand(1).item() < 0.5:
        x, ya, yb, lam = mixup_data(images, targets, alpha=0.2)
    else:
        x, ya, yb, lam = cutmix_data(images, targets, alpha=1.0)
    logits = model(x)
    return mixed_loss(loss_fns, logits, ya, yb, lam)

def make_train_loader(use_sampler):
    if use_sampler:
        sampler = class_balanced_sampler(train_ds, attribute="weather")
        return DataLoader(
            train_ds, batch_size=BATCH, sampler=sampler, num_workers=2,
            worker_init_fn=seed_worker, pin_memory=True,
        )

    generator = torch.Generator(); generator.manual_seed(SEED)
    return DataLoader(
        train_ds, batch_size=BATCH, shuffle=True, num_workers=2,
        worker_init_fn=seed_worker, generator=generator, pin_memory=True,
    )

def train_experiment(spec):
    set_seed(SEED, deterministic=True)
    train_loader = make_train_loader(spec["use_sampler"])
    model, matched_tensors = build_pretrained_vit()
    loss_fns = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level3-{spec['name']}",
        config={
            "backbone": "vit_s16_pretrained",
            "pretrained_source": DEIT_SMALL_URL,
            "matched_tensors": matched_tensors,
            "sampler": "class_balanced(weather)" if spec["use_sampler"] else "none",
            "augmentation": "mixup+cutmix" if spec["use_mix"] else "standard",
            "loss": "cross_entropy",
            "epochs": EPOCHS, "batch": BATCH, "lr": LR,
            "weight_decay": WEIGHT_DECAY, "seed": SEED,
        },
        tags=WANDB_TAGS + [spec["name"]],
    )
    trainer = MultiTaskTrainer(
        model, optim, sched, loss_fns, device,
        TrainConfig(epochs=EPOCHS), logger=logger,
    )

    history = {"train_loss": [], "val_avg_mf1": [], "val_per_mf1": []}
    best_score = -1.0
    best_epoch = 0
    best_state = None
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"level3_{spec['name']}.pth")

    # TODO: step_with_mix 와 trainer.evaluate() 를 사용하여 학습 루프를 작성하세요.
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        num_batches = 0
        pbar = tqdm(train_loader, desc=f"{spec['name']} e{epoch + 1}", leave=False)

        for batch in pbar:
            images = batch["image"].to(device, non_blocking=True)
            targets = {a: batch[a].to(device, non_blocking=True) for a in ATTRIBUTES}
            optim.zero_grad(set_to_none=True)

            with torch.amp.autocast(
                device_type=device.type, enabled=trainer.cfg.amp and device.type == "cuda"
            ):
                if spec["use_mix"]:
                    loss = step_with_mix(model, loss_fns, images, targets)
                else:
                    logits = model(images)
                    loss = sum(loss_fns[a](logits[a], targets[a]) for a in ATTRIBUTES)

            trainer.scaler.scale(loss).backward()
            if trainer.cfg.grad_clip is not None:
                trainer.scaler.unscale_(optim)
                nn.utils.clip_grad_norm_(model.parameters(), trainer.cfg.grad_clip)
            trainer.scaler.step(optim)
            trainer.scaler.update()

            running_loss += loss.item()
            num_batches += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        train_loss = running_loss / max(num_batches, 1)
        val_metrics = trainer.evaluate(val_loader)
        sched.step()

        history["train_loss"].append(train_loss)
        history["val_avg_mf1"].append(val_metrics["avg_macro_f1"])
        history["val_per_mf1"].append(val_metrics["per_macro_f1"])

        log_payload = {
            "epoch": epoch + 1, "train/loss": train_loss,
            "val/avg_macro_f1": val_metrics["avg_macro_f1"],
            "lr": optim.param_groups[0]["lr"],
        }
        for attr, value in val_metrics["per_macro_f1"].items():
            log_payload[f"val/mf1_{attr}"] = value
        logger.log(log_payload, step=epoch)

        if val_metrics["avg_macro_f1"] > best_score:
            best_score = val_metrics["avg_macro_f1"]
            best_epoch = epoch + 1
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            torch.save({
                "state_dict": best_state, "history": history, "experiment": spec,
                "best_val_avg_mf1": best_score, "best_epoch": best_epoch,
                "backbone": "vit_s16_pretrained",
                "pretrained_source": DEIT_SMALL_URL,
            }, checkpoint_path)

        print(
            f"[{spec['name']} {epoch + 1:02d}/{EPOCHS}] "
            f"loss={train_loss:.4f} val_avg_MF1={val_metrics['avg_macro_f1']:.4f} "
            f"per={val_metrics['per_macro_f1']}"
        )

    model.load_state_dict(best_state)
    val_pred, val_probs, val_tgt, _ = collect_predictions(model, val_loader, device)
    report = evaluation_report(val_pred, val_probs, val_tgt)
    print_evaluation_report(report, title=f"Level 3 - {spec['name']}")
    logger.log_evaluation("final", report)

    torch.save({
        "state_dict": best_state, "history": history, "experiment": spec,
        "best_val_avg_mf1": best_score, "best_epoch": best_epoch,
        "backbone": "vit_s16_pretrained", "pretrained_source": DEIT_SMALL_URL,
    }, checkpoint_path)
    logger.finish()

    model = model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        "name": spec["name"], "best_val_avg_mf1": best_score,
        "best_epoch": best_epoch, "checkpoint": checkpoint_path,
    }

def load_experiment(spec):
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"level3_{spec['name']}.pth")
    payload = torch.load(checkpoint_path, map_location="cpu")
    model = vit_small_patch16_224().to(device)
    model.load_state_dict(payload["state_dict"])
    val_pred, val_probs, val_tgt, _ = collect_predictions(model, val_loader, device)
    report = evaluation_report(val_pred, val_probs, val_tgt)
    print_evaluation_report(report, title=f"Level 3 - {spec['name']} (checkpoint)")
    model = model.cpu()
    return {
        "name": spec["name"],
        "best_val_avg_mf1": payload.get("best_val_avg_mf1", report["avg_macro_f1"]),
        "best_epoch": payload.get("best_epoch", 0),
        "checkpoint": checkpoint_path,
    }

results = (
    [train_experiment(spec) for spec in EXPERIMENTS]
    if RUN_TRAINING else [load_experiment(spec) for spec in EXPERIMENTS]
)

In [ ]:
# 전체 실험 중 validation Avg-MF1 이 가장 높은 checkpoint를 Level 5 입력으로 지정.
import shutil

best_result = max(results, key=lambda result: result["best_val_avg_mf1"])
best_path = os.path.join(CHECKPOINT_DIR, "level3_best.pth")
shutil.copy2(best_result["checkpoint"], best_path)

print("Level 3 experiment results:")
for result in results:
    print(
        f"  {result['name']}: best Avg-MF1={result['best_val_avg_mf1']:.4f} "
        f"at epoch {result['best_epoch']}"
    )
print(f"Best checkpoint: {best_path} ({best_result['name']})")

## 분석 (필수)

각 기법에 대해 **속성별 per-class F1 표** 를 작성하세요. 다음 항목을 강조해 주세요.
- 소수 클래스 (foggy / snowy / dawn-dusk) 의 적용 전후 성능 차이.
- 다수 클래스의 회귀 (regression) 발생 여부 — 그 trade-off 가 정당한지 논거.
- Sampling 과 Mixup / CutMix 의 상호작용 — 서로 도움이 되는지 충돌하는지.

In [ ]:
# 다음 Level을 위해 CPU/GPU 메모리를 정리합니다.
import gc
for _name in ["results","best_result","train_loader","val_loader","train_ds","val_ds"]:
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# Level 4 — 심층 분석: Grad-CAM 및 효율성 Trade-off

**목표**: 모델의 *동작 원리* 를 설명하고, FPS와 정확도의 trade-off 를 정량화합니다.

리포트 필수 산출물:
1. **속성별 정규화 Confusion Matrix 3개** — best 모델 기준.
2. **Grad-CAM 패널** — 같은 이미지에 대해 3개 head 가 각각 어디를 보는지 시각화 (예: `rainy + night + city street` 인 이미지).
3. 모든 백본에 대한 **FPS vs Avg-MF1 Pareto plot**.

본 노트북은 학습 노트북이 아니라 분석 노트북이지만, wandb 가 활성화되어 있으면 confusion matrix 이미지·Grad-CAM 패널·FPS 표를 같은 프로젝트의 별도 Run 으로 업로드합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed
from src.utils.transforms import eval_transform
from src.utils.metrics import collect_predictions, evaluation_report, print_evaluation_report, CLASS_NAMES
from src.utils.efficiency import measure_fps
from src.utils.wandb_logger import WandbLogger
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.xai.gradcam import GradCAM
from src.models.vgg import VGG16
from src.models.resnet import resnet18, resnet50
from src.models.vit import vit_small_patch16_224

set_seed(42, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
USE_WANDB = os.environ.get("AUE8088_USE_WANDB", "0") == "1"
if USE_WANDB:
    import wandb
    wandb.login()

WANDB_PROJECT = "aue8088-pa2" if USE_WANDB else None
logger = WandbLogger(project=WANDB_PROJECT, run_name="level4-analysis", tags=["level4", "analysis"])

In [ ]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# Level 3에서 선택된 best ViT-S 모델 로드.
val_ds = BDDAttrDataset("../data/set_a", "val", transform=eval_transform())
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

model = vit_small_patch16_224().to(device)
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "level3_best.pth"), map_location=device)
model.load_state_dict(ckpt["state_dict"])
model.eval()
print(
    f"Loaded {ckpt.get('backbone', 'vit_s16_pretrained')} "
    f"from Level 3 experiment: {ckpt.get('experiment', {})}"
)

In [ ]:
# 속성별 정규화 Confusion Matrix 생성 및 시각화
preds, probs, targets, _ = collect_predictions(model, val_loader, device)
report = evaluation_report(preds, probs, targets)
print_evaluation_report(report, title="Level 4 - Level 3 best model")
logger.log_evaluation("analysis", report)
cms = report["confusion_matrices"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, a in zip(axes, ATTRIBUTES):
    sns.heatmap(cms[a], annot=True, fmt=".2f", cmap="Blues", ax=ax, cbar=False,
                xticklabels=CLASS_NAMES[a], yticklabels=CLASS_NAMES[a])
    ax.set_title(f"{a}")
    ax.set_xlabel("예측 (pred)"); ax.set_ylabel("정답 (true)")
fig.tight_layout()
logger.log_image("analysis/confusion_matrices", fig)
plt.show()

# 속성별 개별 confusion matrix 도 분리해서 업로드
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"analysis/cm_{a}", cms[a], CLASS_NAMES[a])

In [ ]:
# ViT patch embedding의 14x14 feature map을 target layer로 사용.
target_layer = model.patch_embed.proj
gc = GradCAM(model, target_layer)

batch = next(iter(val_loader))
x = batch["image"][:6].to(device)  # 샘플 이미지 6장

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for row, attr in enumerate(ATTRIBUTES):
    # 각 속성 head 의 최대 logit 합을 score 로 사용 → 해당 head 가 "보는" 영역 추출
    cam = gc(x, lambda out, a=attr: out[a].max(dim=-1).values.sum())
    for col in range(6):
        img = x[col].cpu().permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min())
        axes[row, col].imshow(img)
        axes[row, col].imshow(cam[col].cpu().numpy(), cmap="jet", alpha=0.45)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(attr, fontsize=14)
fig.tight_layout()
logger.log_image("analysis/gradcam_panel", fig)
plt.show()

In [ ]:
# FPS 측정 전 XAI 모델의 GPU 메모리를 반환.
del gc
model = model.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# FPS 측정 — 배치 1, 224x224, warm-up 20회 후 200회 평균.
model_fns = {
    "VGG16": VGG16,
    "ResNet-18": resnet18,
    "ResNet-50": resnet50,
    "ViT-S": vit_small_patch16_224,
}
checkpoint_specs = {
    "VGG16": (VGG16, "level1_vgg16.pth"),
    "ResNet-18": (resnet18, "level1_resnet18.pth"),
    "ResNet-50": (resnet50, "level1_resnet50.pth"),
    "ViT-S scratch": (vit_small_patch16_224, "level2_vit_s16_scratch.pth"),
    "ViT-S pretrained": (vit_small_patch16_224, "level2_vit_s16_pretrained.pth"),
}

def checkpoint_state(payload):
    return payload["state_dict"] if isinstance(payload, dict) and "state_dict" in payload else payload

avg_mf1 = {}
for name, (model_fn, filename) in checkpoint_specs.items():
    evaluated_model = model_fn().to(device)
    payload = torch.load(os.path.join(CHECKPOINT_DIR, filename), map_location="cpu")
    evaluated_model.load_state_dict(checkpoint_state(payload))
    eval_preds, eval_probs, eval_targets, _ = collect_predictions(evaluated_model, val_loader, device)
    eval_report = evaluation_report(eval_preds, eval_probs, eval_targets)
    avg_mf1[name] = eval_report["avg_macro_f1"]
    print(f"{name:16s} validation Avg-MF1 = {avg_mf1[name]:.5f}")
    del evaluated_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

fps_by_arch = {}
for name, fn in model_fns.items():
    measured_model = fn().to(device).eval()
    fps = measure_fps(measured_model, device, batch_size=1, n_warmup=20, n_iter=200)
    fps_by_arch[name] = fps
    print(f"{name:12s} FPS = {fps:.2f}")
    del measured_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

fps_rows = [
    ["VGG16", round(fps_by_arch["VGG16"], 2), avg_mf1["VGG16"]],
    ["ResNet-18", round(fps_by_arch["ResNet-18"], 2), avg_mf1["ResNet-18"]],
    ["ResNet-50", round(fps_by_arch["ResNet-50"], 2), avg_mf1["ResNet-50"]],
    ["ViT-S scratch", round(fps_by_arch["ViT-S"], 2), avg_mf1["ViT-S scratch"]],
    ["ViT-S pretrained", round(fps_by_arch["ViT-S"], 2), avg_mf1["ViT-S pretrained"]],
]
logger.log_table("analysis/fps_accuracy", ["backbone", "FPS", "Avg-MF1"], fps_rows)

# TODO: 모든 백본에 대해 FPS (x축) vs Avg-MF1 (y축) Pareto plot 을 작성하세요.
points = [(row[0], row[1], row[2]) for row in fps_rows]
pareto = []
for name, fps, score in points:
    dominated = any(
        other_fps >= fps and other_score >= score
        and (other_fps > fps or other_score > score)
        for other_name, other_fps, other_score in points if other_name != name
    )
    if not dominated:
        pareto.append((name, fps, score))

fig, ax = plt.subplots(figsize=(8, 6))
for name, fps, score in points:
    is_pareto = any(item[0] == name for item in pareto)
    ax.scatter(fps, score, s=90, marker="o" if is_pareto else "x",
               color="tab:red" if is_pareto else "tab:gray")
    ax.annotate(name, (fps, score), xytext=(6, 6), textcoords="offset points")

pareto_sorted = sorted(pareto, key=lambda item: item[1])
ax.plot([item[1] for item in pareto_sorted], [item[2] for item in pareto_sorted],
        color="tab:red", linestyle="--", alpha=0.7, label="Pareto frontier")
ax.set_xlabel("FPS (batch=1, T4)")
ax.set_ylabel("Validation Avg-MF1")
ax.set_title("Efficiency vs Accuracy")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
logger.log_image("analysis/pareto", fig)
plt.show()

In [ ]:
logger.finish()

In [ ]:
# 다음 Level을 위해 CPU/GPU 메모리를 정리합니다.
import gc
for _name in ["model","val_loader","val_ds","preds","probs","targets"]:
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# Level 5 — Data Mining Challenge: *The 1,000-Pick*

**규칙**: Set B (이미지 + 라벨 공개) 에서 **최대 1,000장** 을 선택하여 학습 셋에 추가하고, best 모델을 다시 학습하세요.

> **Set B 의 라벨이 공개되어 있다는 점에 주의**하세요. 본 Level 의 평가 본질은 "*주어진 풀에서 어떤 1,000장이 가장 가치 있는가*" — 즉, 라벨을 알고 있다고 가정한 상태에서의 효율적인 부분집합 선택입니다.

**본 PA에서 가장 큰 비중 (25%)** 을 차지하는 Level 입니다. 어떤 *알고리즘* 으로 1,000장을 골랐는지 — 그 *근거* — 가 변별력의 본진입니다. Curation Report 로 정리합니다.

채점 메트릭:
$$\text{DI} = \frac{\text{Avg-MF1}(\text{본인 picks}) - \text{Avg-MF1}(\text{random picks})}{\text{Avg-MF1}(\text{random picks})}$$

## 검토해 볼 만한 전략

| 전략 | 핵심 아이디어 | Set B 라벨 활용 |
|---|---|---|
| 클래스 균형 (Class Balancing) | Set A 에서 부족한 속성 클래스 (foggy / dawn-dusk 등) 를 채워 넣음 | ✅ 라벨로 직접 필터링 |
| Hard Example Mining | base 모델의 confidence 가 낮은 / 예측이 라벨과 다른 이미지를 우선 선택 | ✅ 모델 예측 vs 정답 비교 |
| 다양성 (Core-Set) | Set B 의 feature space 를 가장 잘 커버하는 부분집합 선택 (k-center / clustering) | 라벨 무관 |
| 결합 커버리지 | 속성 *조합* 의 균형을 맞춤 — 예: (snowy & night), (rainy & residential) | ✅ 라벨로 조합 카운트 |
| Loss 기반 | Set B 이미지에 대한 학습 직전 loss 가 큰 샘플 우선 | ✅ 라벨 필요 |

위 전략들을 결합/응용/대체할 수 있습니다. **Curation Report 에 본인의 의사결정 근거를 명확히 기술** 하세요.

**산출물**: `level5_picks.json` — 선택한 image_id 리스트 (이미지별 메타데이터 포함 가능).

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, evaluation_report, print_evaluation_report, CLASS_NAMES
from src.utils.submission import write_submission
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.models.vit import vit_small_patch16_224

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
USE_WANDB = os.environ.get("AUE8088_USE_WANDB", "0") == "1"
if USE_WANDB:
    import wandb
    wandb.login()

WANDB_PROJECT = "aue8088-pa2" if USE_WANDB else None
STRATEGY_NAME = "hybrid-rare-hard-joint"
WANDB_TAGS    = ["level5", STRATEGY_NAME]
RUN_TRAINING = os.environ.get("AUE8088_RUN_TRAINING", "1") == "1"  # False이면 checkpoint 평가만 함
TA_RANDOM_BASELINE_MF1 = None  # 조교 공지 수치가 나오면 입력

In [ ]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------


In [ ]:
# 1단계 — Level 3 best ViT-S 모델로 Set B 전체를 score.
level3_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "level3_best.pth"), map_location="cpu")
model = vit_small_patch16_224()
model.load_state_dict(level3_ckpt["state_dict"])
model = model.to(device).eval()

set_b = BDDAttrDataset("../data/set_b", split="mining", transform=eval_transform())
loader_b = DataLoader(set_b, batch_size=128, shuffle=False, num_workers=2)

preds_b, probs_b, _, ids_b = collect_predictions(model, loader_b, device)

# 이미지별 불확실성 (uncertainty) 계산: 1 - max-softmax 를 3개 head 평균.
max_probs = np.stack([probs_b[a].max(axis=-1) for a in ATTRIBUTES], axis=1)
uncertainty = 1.0 - max_probs.mean(axis=1)
print(f"unc shape: {uncertainty.shape}, mean={uncertainty.mean():.3f}")
model = model.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# 2단계 — 희소 클래스, 희소 속성 조합, 불확실성, 오분류를 결합한 선별 전략.
import json
from collections import Counter

train_reference = BDDAttrDataset("../data/set_a", "train")
labels_b = {
    "weather": np.array([sample.weather for sample in set_b.samples]),
    "scene": np.array([sample.scene for sample in set_b.samples]),
    "timeofday": np.array([sample.timeofday for sample in set_b.samples]),
}

def minmax(values):
    values = np.asarray(values, dtype=np.float64)
    return (values - values.min()) / (values.max() - values.min() + 1e-12)

# Set A에서 적은 클래스일수록 높은 점수. 세 속성의 희소도를 평균.
attribute_rarity = np.zeros(len(set_b), dtype=np.float64)
for attr in ATTRIBUTES:
    counts = train_reference.class_counts(attr).numpy().astype(np.float64)
    per_sample = 1.0 / np.sqrt(np.maximum(counts[labels_b[attr]], 1.0))
    attribute_rarity += minmax(per_sample) / len(ATTRIBUTES)

# Weather×Scene×Time 조합이 Set A에서 드물수록 높은 점수.
joint_counts = Counter(
    (sample.weather, sample.scene, sample.timeofday) for sample in train_reference.samples
)
joint_rarity = minmax([
    1.0 / np.sqrt(joint_counts.get(
        (sample.weather, sample.scene, sample.timeofday), 0
    ) + 1.0)
    for sample in set_b.samples
])

# 현재 best 모델이 틀린 속성의 비율을 hard-example 신호로 사용.
error_rate = np.mean(np.stack([
    preds_b[attr] != labels_b[attr] for attr in ATTRIBUTES
], axis=1), axis=1)

uncertainty_score = minmax(uncertainty)
selection_score = (
    0.30 * attribute_rarity
    + 0.25 * joint_rarity
    + 0.25 * uncertainty_score
    + 0.20 * error_rate
)

K = 1000
selected_order = np.argsort(-selection_score)[:K]
rng = np.random.default_rng(SEED)
random_order = rng.choice(len(set_b), size=K, replace=False)

def records_from_indices(indices, reason):
    records = []
    for index in indices:
        sample = set_b.samples[int(index)]
        records.append({
            "image_id": sample.image_id,
            "weather": int(sample.weather),
            "scene": int(sample.scene),
            "timeofday": int(sample.timeofday),
            "selection_score": float(selection_score[index]),
            "uncertainty": float(uncertainty_score[index]),
            "attribute_rarity": float(attribute_rarity[index]),
            "joint_rarity": float(joint_rarity[index]),
            "error_rate": float(error_rate[index]),
            "reason": reason,
        })
    return records

picks = records_from_indices(selected_order, STRATEGY_NAME)
random_picks = records_from_indices(random_order, "random-baseline")
picks_path = os.path.join(OUTPUT_DIR, "level5_picks.json")
with open(picks_path, "w") as file:
    json.dump({
        "strategy": (
            "30% attribute rarity + 25% joint rarity + 25% uncertainty "
            "+ 20% attribute error rate"
        ),
        "num_picks": len(picks),
        "picks": picks,
    }, file, indent=2)
print(f"saved {len(picks)} picks to {picks_path}")

In [ ]:
# 3단계 — selected 250/500/1000 ablation과 random-1000 baseline을 동일 조건으로 학습.
val_ds = BDDAttrDataset("../data/set_a", "val", transform=eval_transform())
loader_val = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
FINETUNE_EPOCHS = 25
FINETUNE_LR = 1e-4

def train_with_picks(run_name, pick_records):
    extra = [
        (item["image_id"], item["weather"], item["scene"], item["timeofday"])
        for item in pick_records
    ]
    train_aug = BDDAttrDataset(
        "../data/set_a", "train", transform=train_transform(), extra_picks=extra
    )
    generator = torch.Generator(); generator.manual_seed(SEED)
    loader_tr = DataLoader(
        train_aug, batch_size=64, shuffle=True, num_workers=2,
        worker_init_fn=seed_worker, generator=generator, pin_memory=True,
    )

    set_seed(SEED, deterministic=True)
    trained_model = vit_small_patch16_224()
    trained_model.load_state_dict(level3_ckpt["state_dict"])
    trained_model = trained_model.to(device)
    optim = torch.optim.AdamW(
        trained_model.parameters(), lr=FINETUNE_LR, weight_decay=5e-2
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=FINETUNE_EPOCHS)
    losses = {attr: nn.CrossEntropyLoss() for attr in ATTRIBUTES}
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"level5_{run_name}.pth")

    run_logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level5-{run_name}",
        config={
            "backbone": "vit_s16_level3_best", "strategy": run_name,
            "num_picks": len(pick_records), "epochs": FINETUNE_EPOCHS,
            "batch": 64, "lr": FINETUNE_LR, "seed": SEED,
        },
        tags=["level5", run_name],
    )
    trainer = MultiTaskTrainer(
        trained_model, optim, sched, losses, device,
        TrainConfig(
            epochs=FINETUNE_EPOCHS, checkpoint_path=checkpoint_path,
            checkpoint_metadata={"strategy": run_name, "num_picks": len(pick_records)},
        ), logger=run_logger,
    )
    history = trainer.fit(loader_tr, loader_val)

    # trainer.fit은 정해진 epoch를 모두 수행한 뒤 validation 최고 가중치를 복원한다.
    val_pred, val_probs, val_tgt, _ = collect_predictions(trained_model, loader_val, device)
    report = evaluation_report(val_pred, val_probs, val_tgt)
    print_evaluation_report(report, title=f"Level 5 - {run_name}")
    run_logger.log_evaluation("final", report)
    for attr in ATTRIBUTES:
        counts = Counter(item[attr] for item in pick_records)
        rows = [[CLASS_NAMES[attr][index], counts.get(index, 0)] for index in range(NUM_CLASSES[attr])]
        run_logger.log_table(f"picks/distribution_{attr}", ["class", "count"], rows)

    best_score = history["best_val_avg_mf1"]
    torch.save({
        "state_dict": trained_model.state_dict(), "history": history,
        "strategy": run_name, "num_picks": len(pick_records),
        "best_val_avg_mf1": best_score, "best_epoch": history["best_epoch"],
    }, checkpoint_path)
    run_logger.finish()

    trained_model = trained_model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        "name": run_name, "num_picks": len(pick_records),
        "best_val_avg_mf1": best_score, "checkpoint": checkpoint_path,
    }

run_specs = [
    (f"{STRATEGY_NAME}-250", picks[:250]),
    (f"{STRATEGY_NAME}-500", picks[:500]),
    (f"{STRATEGY_NAME}-1000", picks),
    ("random-1000", random_picks),
]
def load_pick_experiment(run_name, pick_records):
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"level5_{run_name}.pth")
    payload = torch.load(checkpoint_path, map_location="cpu")
    loaded_model = vit_small_patch16_224().to(device)
    loaded_model.load_state_dict(payload["state_dict"])
    val_pred, val_probs, val_tgt, _ = collect_predictions(loaded_model, loader_val, device)
    report = evaluation_report(val_pred, val_probs, val_tgt)
    print_evaluation_report(report, title=f"Level 5 - {run_name} (checkpoint)")
    loaded_model = loaded_model.cpu()
    return {
        "name": run_name, "num_picks": len(pick_records),
        "best_val_avg_mf1": payload.get(
            "best_val_avg_mf1", payload.get("final_val_avg_mf1", report["avg_macro_f1"])
        ),
        "checkpoint": checkpoint_path,
    }

results = (
    [train_with_picks(name, records) for name, records in run_specs]
    if RUN_TRAINING else [load_pick_experiment(name, records) for name, records in run_specs]
)

selected_1000 = next(result for result in results if result["name"].endswith("-1000") and result["name"] != "random-1000")
random_1000 = next(result for result in results if result["name"] == "random-1000")
self_random_di = (
    selected_1000["best_val_avg_mf1"] - random_1000["best_val_avg_mf1"]
) / max(random_1000["best_val_avg_mf1"], 1e-12)
official_di = None
if TA_RANDOM_BASELINE_MF1 is not None:
    official_di = (
        selected_1000["best_val_avg_mf1"] - TA_RANDOM_BASELINE_MF1
    ) / TA_RANDOM_BASELINE_MF1

summary_logger = WandbLogger(
    project=WANDB_PROJECT, run_name="level5-summary", tags=["level5", "summary"]
)
summary_logger.log_table(
    "level5/results", ["strategy", "num_picks", "best_avg_mf1"],
    [[result["name"], result["num_picks"], result["best_val_avg_mf1"]] for result in results],
)
summary_logger.log({"level5/self_random_di": self_random_di})
if official_di is not None:
    summary_logger.log({"level5/official_di": official_di})
summary_logger.finish()

import shutil
final_checkpoint = os.path.join(CHECKPOINT_DIR, "level5_final.pth")
shutil.copy2(selected_1000["checkpoint"], final_checkpoint)
print(f"Self-computed random baseline DI = {self_random_di:.4f}")
if official_di is None:
    print("Official DI pending: enter the TA-provided baseline in TA_RANDOM_BASELINE_MF1.")
else:
    print(f"Official TA-baseline DI = {official_di:.4f}")
print(f"Final checkpoint: {final_checkpoint}")

## Curation Report — 필수

Final PPT 에 다음을 포함하세요.
- **선별 알고리즘** (의사코드 또는 1페이지 다이어그램).
- 본인 picks 1,000장의 **분포** — (예측된) weather × scene × timeofday — 를 heatmap 또는 stacked bar 로 시각화.
- **Random-1000 baseline** 결과와 본인의 **DI score** 비교.
- **Ablation**: 250 / 500 / 1000 장을 골랐을 때의 변화 — 추가 데이터의 한계 효용이 보이는지 확인.

여러 전략을 시험했다면 wandb 의 같은 프로젝트에 `STRATEGY_NAME` 만 바꿔서 별도 Run 으로 누적하세요. 학습 곡선·분포·DI score 비교가 한 페이지에 모입니다.

In [ ]:
# 제출 산출물이 모두 생성되었는지 확인합니다.
required_checkpoints = [
    "level1_vgg16.pth", "level1_resnet18.pth", "level1_resnet50.pth",
    "level1_resnet18-w211.pth", "level2_vit_s16_scratch.pth",
    "level2_vit_s16_pretrained.pth", "level3_best.pth",
    "level5_hybrid-rare-hard-joint-1000.pth", "level5_random-1000.pth",
    "level5_final.pth",
]
missing = [name for name in required_checkpoints if not os.path.exists(os.path.join(CHECKPOINT_DIR, name))]
picks_path = os.path.join(OUTPUT_DIR, "level5_picks.json")
assert not missing, f"Missing checkpoints: {missing}"
assert os.path.exists(picks_path), f"Missing artifact: {picks_path}"
print(f"Verified {len(required_checkpoints)} checkpoints")
print(f"Curation artifact: {picks_path}")
